# 3_16 Train sequence predictor RNN (next-step GRU)

Lean runnable training notebook (no slide deck).

Many-to-many GRU predicts the **next 2D point** along orbit / wave / spiral
trajectories. Context `x_1..x_{T-1}` → targets `x_2..x_T`. Loss = MSE.

| Piece | Role |
|-------|------|
| data | curved trajectories in `R^2` |
| model | `GRU(2→32)` + `Linear(32→2)` at every step |
| loss | `F.mse_loss` |
| loop | Adam + mini-batches; monitor val; report test once |


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(47)
np.random.seed(47)

T, N_SEQ, NOISE = 24, 240, 0.06
EPOCHS, LR, BATCH_SIZE = 40, 0.03, 32


def make_orbit(rng):
    t = np.linspace(0, 2 * np.pi, T, endpoint=False)
    a = 0.9 + 0.2 * rng.randn()
    b = 0.55 + 0.15 * rng.randn()
    phase = rng.uniform(0, 2 * np.pi)
    rot = rng.uniform(0, 2 * np.pi)
    xy = np.stack([a * np.cos(t + phase), b * np.sin(t + phase)], axis=1)
    c, s = np.cos(rot), np.sin(rot)
    R = np.array([[c, -s], [s, c]], dtype=np.float32)
    xy = xy.astype(np.float32) @ R.T
    return xy + NOISE * rng.randn(T, 2).astype(np.float32)


def make_wave(rng):
    t = np.linspace(-1.2, 1.2, T)
    amp = 0.45 + 0.15 * abs(rng.randn())
    freq = 1.5 + 0.5 * rng.randn()
    phase = rng.uniform(0, 2 * np.pi)
    xy = np.stack([t, amp * np.sin(freq * np.pi * t + phase)], axis=1)
    return xy.astype(np.float32) + NOISE * rng.randn(T, 2).astype(np.float32)


def make_spiral(rng):
    t = np.linspace(0, 2.2 * np.pi, T)
    r = np.linspace(0.25, 1.0, T) * (0.9 + 0.2 * rng.randn())
    phase = rng.uniform(0, 2 * np.pi)
    xy = np.stack([r * np.cos(t + phase), r * np.sin(t + phase)], axis=1)
    return xy.astype(np.float32) + NOISE * rng.randn(T, 2).astype(np.float32)


MAKERS = [make_orbit, make_wave, make_spiral]
seqs = []
for i in range(N_SEQ):
    maker = MAKERS[i % len(MAKERS)]
    rng = np.random.RandomState(2000 + i)
    seqs.append(maker(rng))
X = np.stack(seqs, axis=0)  # (N, T, 2)

# Next-step supervision (teacher forcing):
X_ctx = X[:, :-1, :]   # (N, T-1, 2)  inputs  x_1..x_{T-1}
Y_nxt = X[:,  1:, :]   # (N, T-1, 2)  targets x_2..x_T

perm = np.random.permutation(len(X))
X_ctx, Y_nxt = X_ctx[perm], Y_nxt[perm]
print(X_ctx.shape, Y_nxt.shape)


## Model

GRU hidden state at each step → predicted next `(x, y)`.


In [ ]:
class NextStepRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.GRU(2, 32, batch_first=True)
        self.fc = nn.Linear(32, 2)

    def forward(self, x):       # x: (B, T-1, 2)
        h, _ = self.rnn(x)      # h: (B, T-1, 32)
        return self.fc(h)       # (B, T-1, 2) = xhat_{2..T}


def dramatic_init_(model: NextStepRNN) -> None:
    """Bad head so predictions start far from truth."""
    with torch.no_grad():
        model.fc.weight.normal_(0.0, 1.2)
        model.fc.bias.copy_(torch.tensor([1.5, -1.2]))


n = len(X_ctx)
n_train, n_val = int(0.60 * n), int(0.20 * n)

def split3(arr):
    return (
        arr[:n_train],
        arr[n_train:n_train + n_val],
        arr[n_train + n_val:],
    )

C_tr, C_va, C_te = split3(X_ctx)
Y_tr, Y_va, Y_te = split3(Y_nxt)

C_tr_t, Y_tr_t = torch.from_numpy(C_tr), torch.from_numpy(Y_tr)
C_va_t, Y_va_t = torch.from_numpy(C_va), torch.from_numpy(Y_va)
C_te_t, Y_te_t = torch.from_numpy(C_te), torch.from_numpy(Y_te)

train_loader = DataLoader(TensorDataset(C_tr_t, Y_tr_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(C_va_t, Y_va_t), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TensorDataset(C_te_t, Y_te_t), batch_size=BATCH_SIZE, shuffle=False)

torch.manual_seed(7)
model = NextStepRNN()
dramatic_init_(model)
opt = torch.optim.Adam(model.parameters(), lr=LR)
print(model)


## Training loop


In [ ]:
def run_epoch(loader, train=True):
    model.train(train)
    total, n_ex = 0.0, 0
    for xb, yb in loader:
        loss = F.mse_loss(model(xb), yb)
        if train:
            opt.zero_grad()
            loss.backward()
            opt.step()
        total += loss.item() * len(yb)
        n_ex += len(yb)
    return total / n_ex


for epoch in range(EPOCHS):
    tr = run_epoch(train_loader, train=True)
    with torch.no_grad():
        va = run_epoch(val_loader, train=False)
    if epoch == 0 or (epoch + 1) % 10 == 0:
        print(f"epoch {epoch+1:02d}/{EPOCHS}  train_mse={tr:.4f}  val_mse={va:.4f}")


## Inference + one-shot test report


In [ ]:
model.eval()
with torch.no_grad():
    yhat = model(C_te_t[0:1])[0]
    ytrue = Y_te_t[0]
    print("one-seq MSE:", float((yhat - ytrue).pow(2).mean()))
    test_mse = run_epoch(test_loader, train=False)
print(f"test MSE={test_mse:.4f}")
# Do not tune LR / epochs on this number.
